# 01 — Data Analysis

Exploratory analysis of the raw Digikala products and comments datasets.

This notebook is intentionally descriptive only. Data cleaning logic lives in
`src/rag/preprocessing/datasets.py`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

products = pd.read_csv(
    RAW_DIR / "digikala-products.csv",
    low_memory=False,
)

comments = pd.read_csv(
    RAW_DIR / "digikala-comments.csv",
    low_memory=False,
)

print("Products:", products.shape)
print("Comments:", comments.shape)

## Schema and data quality

In [ ]:
display(products.head())
display(comments.head())

print("Product duplicate rows:", products.duplicated().sum())
print("Product duplicate IDs:", products["id"].duplicated().sum())
print("Comment duplicate rows:", comments.duplicated().sum())
print("Comment duplicate IDs:", comments["id"].duplicated().sum())

In [ ]:
product_missing = (
    products.isna().mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_percent")
)

comment_missing = (
    comments.isna().mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_percent")
)

display(product_missing.head(15).to_frame())
display(comment_missing.head(15).to_frame())

## Ratings and recommendation labels

In [ ]:
ax = products["Rate"].dropna().plot(
    kind="hist",
    bins=30,
    figsize=(8, 4),
    title="Product rating distribution",
)
ax.set_xlabel("Product Rate")
plt.tight_layout()
plt.show()

comment_rates = (
    comments["rate"]
    .round()
    .value_counts()
    .sort_index()
)

ax = comment_rates.plot(
    kind="bar",
    figsize=(7, 4),
    title="Comment rating distribution",
)
ax.set_xlabel("Rating")
ax.set_ylabel("Comments")
plt.tight_layout()
plt.show()

In [ ]:
recommendation_counts = (
    comments["recommendation_status"]
    .value_counts(dropna=False)
)

display(
    recommendation_counts
    .rename("count")
    .to_frame()
)

display(
    pd.crosstab(
        comments["rate"],
        comments["recommendation_status"],
        normalize="index",
    ).round(3)
)

## Review text length

In [ ]:
text_length = (
    comments["body"]
    .fillna("")
    .str.len()
)

print(text_length.describe())

ax = text_length.clip(upper=1000).plot(
    kind="hist",
    bins=50,
    figsize=(8, 4),
    title="Review body length (clipped at 1000 chars)",
)
ax.set_xlabel("Characters")
plt.tight_layout()
plt.show()